# We recommend performing UMLS knowledge retrieval using APIs

In [5]:
!pip install umls-api

  Preparing metadata (setup.py) ... done
  Created wheel for umls-api: filename=umls_api-0.1.0-py3-none-any.whl size=3384 sha256=0834dbcd29a60375621e57da100e2834024f38e471bfde2b7aad52588b4ba9da
  Stored in directory: /home/wxy/.cache/pip/wheels/40/d1/1b/52cfe03978dfc09cc52fc981a7a85182ce1891f42ce5af30e9
Successfully built umls-api
  Attempting uninstall: cachetools
    Found existing installation: cachetools 5.3.0
    Uninstalling cachetools-5.3.0:
      Successfully uninstalled cachetools-5.3.0


In [6]:
import numpy as np
import torch
import os
import pickle
import pandas as pd
import requests
from lxml.html import fromstring
from cachetools import cached, TTLCache
import umls_api


abbr={'RQ':'is related and possibly synonymous with',
      'SY':'is asserted to be synonymy of',
      'RN':'denotes a narrower concept compared with',
      'RB':'denotes a broader concept compared with',
      'QB':'can be qualified by',
      'PAR':'include',
      'CHD':'is part of (belongs to)',    
      }


TTL_7HRS = TTLCache(maxsize=2, ttl=25200)

def cui2name(CUI):
    res = umls_api.API(api_key='72d06e11-2fa4-4bf1-b702-ee2d852038a7').get_cui(cui=CUI)
    return res['result']['name']
def link2name(link):
    res = umls_api.API(api_key='72d06e11-2fa4-4bf1-b702-ee2d852038a7').get_snomedct(link=link)
    return res['result']['name']

class Auth:
    def __init__(self, api_key):
        self._api_key = api_key

    @cached(TTL_7HRS)
    def get_single_use_service_ticket(self):
        url = 'https://utslogin.nlm.nih.gov/cas/v1/api-key'
        headers = {
            'Content-type': 'application/x-www-form-urlencoded',
            'Accept': 'text/plain',
            'User-Agent': 'python'
        }
        resp = requests.post(
            url, data={'apikey': self._api_key}, headers=headers
        )
        resp.raise_for_status()
        html = fromstring(resp.text)
        ticket_granting_ticket_url = html.xpath('//form/@action')[0]

        resp = requests.post(
            ticket_granting_ticket_url,
            data={'service': 'http://umlsks.nlm.nih.gov'},
            headers=headers
        )
        resp.raise_for_status()
        single_use_service_ticket = resp.text
        return single_use_service_ticket

class API:
    BASE_URL = 'https://uts-ws.nlm.nih.gov/rest'
                # https://uts-ws.nlm.nih.gov/rest/content/current/CUI/C0155502/definitions?apiKey=YOUR_APIKEY

    def __init__(self, *, api_key, version='current'):
        self._auth = Auth(api_key=api_key)
        self._version = version
        self.api_key=api_key
        self.foreign=['SCTSPA','MSHPOR','MSHSPA','MSHCZE','MSHSWE','MSHNOR']

    def get_cui(self, cui):
        url = f'{self.BASE_URL}/content/{self._version}/CUI/{cui}'
        return self._get(url=url)
    
    def get_name(self, cui):
        url = f'{self.BASE_URL}/content/{self._version}/CUI/{cui}'
        return self._get(url=url)['result']['name']
    
    def get_def(self, cui):
        defi=[]
        url = f'{self.BASE_URL}/content/{self._version}/CUI/{cui}/definitions?apiKey='+self.api_key
        try:
            ret = self._get(url=url)['result']
            for source in ret:
                if source['rootSource'] not in self.foreign:
                    defi.append(source['value'])
            res=defi[0]
            for r in defi:
                if len(r)< len(res):
                    res=r
            return res
        except:
            return 'HTTPError'
        
    def get_defall(self, cui):
        url = f'{self.BASE_URL}/content/{self._version}/CUI/{cui}/definitions?apiKey='+self.api_key
        return self._get(url=url)['result']
    def get_rel(self, cui):
        url = f'{self.BASE_URL}/content/{self._version}/CUI/{cui}/relations?apiKey='+self.api_key
        total= self._get(url=url)['result']
        r=[]
        print('Query CUI:',cui)
        for info in total:
            source=info['rootSource']
            if len(source)!=6 or source=='MEDCIN':
                if source=='KCD5':
                    continue
                try:
                    head=info['relatedFromIdName'] 
                except KeyError:
                    # head='XXX '
                    head=cui2name(cui)
                try:
                # if hasattr(info, "relatedIdName"):
                    tail=info["relatedIdName"]
                    print('relatedId:',tail)
                except KeyError:
                    print('relatedId:',info["relatedId"])
                    if info["relatedId"].split('/')[-2]=='CUI':
                        tail=cui2name(info["relatedId"].split('/')[-1])
                    else:
                        tail=link2name(info["relatedId"])
                try:
                    rel_label=info['additionalRelationLabel']
                    if rel_label=="":
                        rel_label=info['relationLabel']
                    print('RelationLabel:',rel_label)
                except KeyError:
                    print('RelationLabel:',info['relationLabel'])
                    rel_label=info['relationLabel']
                if rel_label=='RO':
                    continue
                if rel_label in abbr.keys():
                    rel_label=abbr[rel_label]
                if rel_label=="inverse_isa":
                    rel=' '.join([tail,' is a ',head])
                else:
                    rel=' '.join([head,rel_label,tail])

                r.append([rel,source])
        return r

            
    
    def get_cui_code(self, keyword):
        url = f'{self.BASE_URL}/search/{self._version}/?string={keyword}'
        candidates= self._get(url=url)['result']['results']
        for i in range(3):
            cui=candidates[i]['ui']
            name=candidates[i]['name']
            print(f'No.{i+1} => CUI = {cui}, Concept Name: {name}')
        return candidates[0]['ui']

    def _get(self, url):
        ticket = self._auth.get_single_use_service_ticket()
        resp = requests.get(url, params={'ticket': ticket})
        resp.raise_for_status()
        return resp.json()

# Definition Retrieval

In [10]:
CUI='C2712342'
cui_name= API(api_key='72d06e11-2fa4-4bf1-b702-ee2d852038a7').get_name(cui=CUI)
cui_def = API(api_key='72d06e11-2fa4-4bf1-b702-ee2d852038a7').get_def(cui=CUI)
print(": ".join([CUI, cui_name, cui_def]))

C2712342: Bladder stone (substance): A concretion in the urinary bladder.


# Relationship Retrieval

In [11]:
cui_rel = API(api_key='72d06e11-2fa4-4bf1-b702-ee2d852038a7').get_rel(cui=CUI)
print(cui_rel)

Query CUI: C2712342
relatedId: Urinary bladder calculus specimen
RelationLabel: specimen_substance_of
relatedId: Urinary bladder stone (disorder)
RelationLabel: RO
relatedId: Analysis of calculus of urinary bladder
RelationLabel: component_of
relatedId: Urinary Stone
RelationLabel: isa
relatedId: Urinary system calculus
RelationLabel: isa
[['Bladder stone specimen_substance_of Urinary bladder calculus specimen', 'SNOMEDCT_US'], ['Bladder stone component_of Analysis of calculus of urinary bladder', 'SNOMEDCT_US'], ['Urinary Bladder Stone isa Urinary Stone', 'NCI'], ['Bladder stone isa Urinary system calculus', 'SNOMEDCT_US']]
